In [2]:

# EfficientNet-B0 — AdaRound Quantization (PTQ)


import os, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0
from sklearn.model_selection import train_test_split


# Dataset & DataLoaders

transform = transforms.Compose([
    transforms.Resize((224,224), antialias=True),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

data_dir = "DIR/horse/sapi279d-test_workspace/train"  # Change path
dataset = datasets.ImageFolder(root=data_dir, transform=transform)

targets = np.array(dataset.targets)
train_idx, val_idx, test_idx = [], [], []
for class_id in np.unique(targets):
    idxs = np.where(targets == class_id)[0]
    train_ids, temp_ids = train_test_split(idxs, test_size=0.15,
                                           random_state=42, shuffle=True)
    val_ids, test_ids = train_test_split(temp_ids, test_size=1/3,
                                         random_state=42, shuffle=True)
    train_idx.extend(train_ids); val_idx.extend(val_ids); test_idx.extend(test_ids)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)
test_dataset  = Subset(dataset, test_idx)

batch_size = 128
num_workers = min(8, os.cpu_count() or 1)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=num_workers, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, pin_memory=True)
print(f"Total images: {len(dataset)}, Classes: {len(dataset.classes)}")
print("DataLoaders ready")


# Load trained FP32 EfficientNet-B0

state_dict = torch.load("efficientnet_b0_weights.pth", map_location="cpu")
# Strip `_orig_mod.` prefix if present
new_state_dict = {k.replace("_orig_mod.", ""): v for k,v in state_dict.items()}

num_classes = len(dataset.classes)  # 200
model_fp32 = efficientnet_b0(weights=None)
model_fp32.classifier[1] = nn.Linear(model_fp32.classifier[1].in_features, num_classes)
model_fp32.load_state_dict(new_state_dict, strict=True)
model_fp32.eval()
print("FP32 EfficientNet-B0 loaded")


# AdaRound helpers

def adaround_weight_tensor(weight, n_bits=8, num_iters=500, lr=1e-2):
    w = weight.detach().to(torch.float32).clone()
    wmin, wmax = w.min(), w.max()
    scale = (wmax - wmin) / (2 ** n_bits - 1 + 1e-12)

    alpha = torch.zeros_like(w, requires_grad=True)
    opt = torch.optim.Adam([alpha], lr=lr)

    with torch.enable_grad():
        for _ in range(num_iters):
            opt.zero_grad()
            soft = torch.sigmoid(alpha)
            w_q = torch.floor(w / (scale + 1e-12)) + soft
            soft_w = w_q * scale
            loss = torch.nn.functional.mse_loss(soft_w, w)
            loss.backward()
            opt.step()
            alpha = alpha.detach().requires_grad_()

    with torch.no_grad():
        hard = (torch.sigmoid(alpha) > 0.5).to(w.dtype)
        w_q = torch.floor(w / (scale + 1e-12)) + hard
        q_w = (w_q * scale).to(weight.dtype)
    return q_w

def apply_adaround(model, n_bits=8, num_iters=500, lr=1e-2):
    m = copy.deepcopy(model).cpu().eval()
    for name, module in m.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)) and module.weight is not None:
            print(f"AdaRounding: {name}.weight")
            q_w = adaround_weight_tensor(module.weight, n_bits=n_bits, num_iters=num_iters, lr=lr)
            module.weight = nn.Parameter(q_w)
    return m


# Evaluation utilities

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.inference_mode():
        for images, labels in loader:
            outputs = model(images)
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

def get_model_size(model, filename="__temp__.pth"):
    torch.save(model.state_dict(), filename)
    size_mb = os.path.getsize(filename) / 1e6
    os.remove(filename)
    return size_mb

def benchmark(model, loader, num_batches=50):
    model.eval()
    start = time.time()
    with torch.inference_mode():
        for i, (images, _) in enumerate(loader):
            if i >= num_batches:
                break
            _ = model(images)
    end = time.time()
    total_imgs = num_batches * loader.batch_size
    total_t = end - start
    ms_per_batch = (total_t / num_batches) * 1000.0
    ms_per_image = (total_t / total_imgs) * 1000.0
    return ms_per_batch, ms_per_image


# Run AdaRound & Report

model_ada = apply_adaround(model_fp32, n_bits=8, num_iters=500, lr=1e-2)

acc_fp32 = evaluate(model_fp32, test_loader)
acc_ada  = evaluate(model_ada,  test_loader)
print(f"FP32 Accuracy: {acc_fp32:.2f}%")
print(f"AdaRound INT8 Accuracy: {acc_ada:.2f}%")

fp32_size = get_model_size(model_fp32)
ada_size  = get_model_size(model_ada)
print(f"FP32 size: {fp32_size:.2f} MB | AdaRound size: {ada_size:.2f} MB")

batch_ms_fp32, img_ms_fp32 = benchmark(model_fp32, test_loader)
batch_ms_ada,  img_ms_ada  = benchmark(model_ada,  test_loader)
print(f"FP32 Inference: {batch_ms_fp32:.2f} ms/batch | {img_ms_fp32:.2f} ms/image")
print(f"AdaRound Inference: {batch_ms_ada:.2f} ms/batch | {img_ms_ada:.2f} ms/image")


Total images: 100000, Classes: 200
DataLoaders ready
FP32 EfficientNet-B0 loaded
AdaRounding: features.0.0.weight
AdaRounding: features.1.0.block.0.0.weight
AdaRounding: features.1.0.block.1.fc1.weight
AdaRounding: features.1.0.block.1.fc2.weight
AdaRounding: features.1.0.block.2.0.weight
AdaRounding: features.2.0.block.0.0.weight
AdaRounding: features.2.0.block.1.0.weight
AdaRounding: features.2.0.block.2.fc1.weight
AdaRounding: features.2.0.block.2.fc2.weight
AdaRounding: features.2.0.block.3.0.weight
AdaRounding: features.2.1.block.0.0.weight
AdaRounding: features.2.1.block.1.0.weight
AdaRounding: features.2.1.block.2.fc1.weight
AdaRounding: features.2.1.block.2.fc2.weight
AdaRounding: features.2.1.block.3.0.weight
AdaRounding: features.3.0.block.0.0.weight
AdaRounding: features.3.0.block.1.0.weight
AdaRounding: features.3.0.block.2.fc1.weight
AdaRounding: features.3.0.block.2.fc2.weight
AdaRounding: features.3.0.block.3.0.weight
AdaRounding: features.3.1.block.0.0.weight
AdaRoundin